<a href="https://colab.research.google.com/github/Soubhik-2005/Sleep-stage-classification-/blob/main/Sleep_Stage_Classification_Using_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys

print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [ ]:
import tensorflow as tf

print("GPU available:", tf.config.list_physical_devices('GPU'))

GPU available: []


In [ ]:
import tensorflow as tf

print("GPU available:", tf.config.list_physical_devices('GPU'))

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 92.0 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [ ]:
!pip install mne

In [ ]:
import mne

print("MNE version:", mne.__version__)

MNE version: 1.13.2


In [ ]:
import mne

data_path = mne.datasets.sleep_physionet.age.fetch_data(
    subjects=[0],
    recording=[0]
)

print(data_path)

Using default location ~/mne_data for PHYSIONET_SLEEP...
Creating /root/mne_data
[]


In [ ]:
from mne.datasets.sleep_physionet.age import fetch_data

files = fetch_data(
    subjects=[0],
    recording=[1]
)

print(files)

Using default location ~/mne_data for PHYSIONET_SLEEP...


  0%|                                              | 0.00/48.3M [00:00<?, ?B/s]

  0%|                                              | 0.00/4.62k [00:00<?, ?B/s]

Download complete in 08m28s (46.1 MB)
[['/root/mne_data/physionet-sleep-data/SC4001E0-PSG.edf', '/root/mne_data/physionet-sleep-data/SC4001EC-Hypnogram.edf']]


In [ ]:
import mne

psg_file = files[0][0]

raw = mne.io.read_raw_edf(
    psg_file,
    preload=False
)

print(raw)

Extracting EDF parameters from /root/mne_data/physionet-sleep-data/SC4001E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
<RawEDF | SC4001E0-PSG.edf, 7 x 7950000 (79500.0 s), ~10 KiB, data not loaded>


/tmp/ipykernel_3994/1051219373.py:5: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/1051219373.py:5: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/1051219373.py:5: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(


In [ ]:
print(raw.ch_names)

['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']


In [ ]:
print(raw.info["sfreq"])

100.0


In [ ]:
eeg_data = raw.get_data(picks=["EEG Fpz-Cz"])

print("Shape:", eeg_data.shape)

Shape: (1, 7950000)


In [ ]:
eeg = eeg_data[0]

print("Shape:", eeg.shape)

Shape: (7950000,)


In [ ]:
print(eeg[:20])

[ 5.01684982e-06 -2.57875458e-06  1.35970696e-06 -2.39120879e-06
 -5.20439560e-06 -7.45494505e-06 -8.29890110e-06 -6.89230769e-06
 -3.70402930e-06 -9.84615385e-07  2.20366300e-06  2.01611722e-06
  1.64102564e-06 -2.67252747e-06 -7.97069597e-07  1.35970696e-06
  2.86007326e-06 -2.20366300e-06  5.15750916e-07  5.86080586e-06]


In [ ]:
sampling_rate = int(raw.info["sfreq"])
epoch_duration = 30

samples_per_epoch = sampling_rate * epoch_duration
num_epochs = len(eeg) // samples_per_epoch

print("Samples per epoch:", samples_per_epoch)
print("Number of complete epochs:", num_epochs)

Samples per epoch: 3000
Number of complete epochs: 2650


In [ ]:
hypnogram_file = files[0][1]

print(hypnogram_file)

/root/mne_data/physionet-sleep-data/SC4001EC-Hypnogram.edf


In [ ]:
annotations = mne.read_annotations(hypnogram_file)

print(annotations)

<Annotations | 154 segments: Sleep stage 1 (24), Sleep stage 2 (40), Sleep ...>


In [ ]:
print(annotations.description)

['Sleep stage W' 'Sleep stage 1' 'Sleep stage 2' 'Sleep stage 3'
 'Sleep stage 2' 'Sleep stage 3' 'Sleep stage 4' 'Sleep stage 3'
 'Sleep stage 4' 'Sleep stage 3' 'Sleep stage 4' 'Sleep stage 3'
 'Sleep stage 4' 'Sleep stage W' 'Sleep stage 3' 'Sleep stage 2'
 'Sleep stage 3' 'Sleep stage 4' 'Sleep stage 3' 'Sleep stage 4'
 'Sleep stage 2' 'Sleep stage 3' 'Sleep stage 2' 'Sleep stage 3'
 'Sleep stage 1' 'Sleep stage 2' 'Sleep stage 3' 'Sleep stage 2'
 'Sleep stage 3' 'Sleep stage 2' 'Sleep stage 3' 'Sleep stage 4'
 'Sleep stage 3' 'Sleep stage 2' 'Sleep stage 3' 'Sleep stage 4'
 'Sleep stage 3' 'Sleep stage 4' 'Sleep stage 1' 'Sleep stage 3'
 'Sleep stage 2' 'Sleep stage 3' 'Sleep stage 2' 'Sleep stage 3'
 'Sleep stage R' 'Sleep stage 1' 'Sleep stage 2' 'Sleep stage 3'
 'Sleep stage 1' 'Sleep stage 2' 'Sleep stage 1' 'Sleep stage 2'
 'Sleep stage 3' 'Sleep stage 2' 'Sleep stage 3' 'Sleep stage 4'
 'Sleep stage 3' 'Sleep stage 4' 'Sleep stage 3' 'Sleep stage 4'
 'Sleep stage 3' 'Sleep s

In [ ]:
from collections import Counter

stage_counts = Counter(annotations.description)

print(stage_counts)

Counter({'Sleep stage 3': 48, 'Sleep stage 2': 40, 'Sleep stage 1': 24, 'Sleep stage 4': 23, 'Sleep stage W': 12, 'Sleep stage R': 6, 'Sleep stage ?': 1})


In [ ]:
for i in range(10):
    print(
        i,
        "Onset:", annotations.onset[i],
        "Duration:", annotations.duration[i],
        "Stage:", annotations.description[i]
    )

0 Onset: 0.0 Duration: 30630.0 Stage: Sleep stage W
1 Onset: 30630.0 Duration: 120.0 Stage: Sleep stage 1
2 Onset: 30750.0 Duration: 390.0 Stage: Sleep stage 2
3 Onset: 31140.0 Duration: 30.0 Stage: Sleep stage 3
4 Onset: 31170.0 Duration: 30.0 Stage: Sleep stage 2
5 Onset: 31200.0 Duration: 150.0 Stage: Sleep stage 3
6 Onset: 31350.0 Duration: 30.0 Stage: Sleep stage 4
7 Onset: 31380.0 Duration: 60.0 Stage: Sleep stage 3
8 Onset: 31440.0 Duration: 60.0 Stage: Sleep stage 4
9 Onset: 31500.0 Duration: 30.0 Stage: Sleep stage 3


In [ ]:
print("Sampling rate:", sampling_rate)
print("Samples per epoch:", samples_per_epoch)

print("\nFirst 10 annotation durations:")
print(annotations.duration[:10])

print("\nAre the durations multiples of 30 seconds?")
print(annotations.duration[:10] % 30)

Sampling rate: 100
Samples per epoch: 3000

First 10 annotation durations:
[3.063e+04 1.200e+02 3.900e+02 3.000e+01 3.000e+01 1.500e+02 3.000e+01
 6.000e+01 6.000e+01 3.000e+01]

Are the durations multiples of 30 seconds?
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [ ]:
print("EEG recording duration:", len(eeg) / sampling_rate, "seconds")

print("First annotation onset:", annotations.onset[0], "seconds")

print(
    "Last annotation end:",
    annotations.onset[-1] + annotations.duration[-1],
    "seconds"
)

EEG recording duration: 79500.0 seconds
First annotation onset: 0.0 seconds
Last annotation end: 86400.0 seconds


In [ ]:
import numpy as np

# Sleep-stage → numerical label
stage_mapping = {
    "Sleep stage W": 0,
    "Sleep stage 1": 1,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,   # Combine stage 3 and 4 → N3
    "Sleep stage R": 4
}

X = []
y = []

for onset, duration, description in zip(
    annotations.onset,
    annotations.duration,
    annotations.description
):
    # Ignore unknown annotations
    if description not in stage_mapping:
        continue

    label = stage_mapping[description]

    # Number of complete 30-second epochs
    num_segments = int(duration // epoch_duration)

    for j in range(num_segments):
        start_time = onset + j * epoch_duration
        end_time = start_time + epoch_duration

        # Don't go beyond EEG recording
        if end_time > len(eeg) / sampling_rate:
            continue

        start_sample = int(start_time * sampling_rate)
        end_sample = start_sample + samples_per_epoch

        epoch = eeg[start_sample:end_sample]

        if len(epoch) == samples_per_epoch:
            X.append(epoch)
            y.append(label)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2650, 3000)
y shape: (2650,)


In [ ]:
from collections import Counter

print(Counter(y))

Counter({np.int64(0): 1997, np.int64(2): 250, np.int64(3): 220, np.int64(4): 125, np.int64(1): 58})


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (2120, 3000)
X_test : (530, 3000)
y_train: (2120,)
y_test : (530,)


In [ ]:
mean = X_train.mean()
std = X_train.std()

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

print("Training mean:", X_train.mean())
print("Training std :", X_train.std())

print("Test mean:", X_test.mean())
print("Test std :", X_test.std())

Training mean: 2.3656827719685726e-17
Training std : 1.0000000000000007
Test mean: 0.00016495413036698314
Test std : 0.9925345371704533


In [ ]:
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (2120, 3000, 1)
X_test shape : (530, 3000, 1)


In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

model = Sequential([
    Conv1D(32, kernel_size=7, activation="relu", input_shape=(3000, 1)),
    MaxPooling1D(pool_size=2),

    Conv1D(64, kernel_size=5, activation="relu"),
    MaxPooling1D(pool_size=2),

    Flatten(),

    Dense(64, activation="relu"),
    Dropout(0.5),

    Dense(5, activation="softmax")
])

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 2994, 32)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 1497, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 1493, 64)       │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 746, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 47744)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     3,055,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,066,565 (11.70 MB)

 Trainable params: 3,066,565 (11.70 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout

model = Sequential([
    Input(shape=(3000, 1)),

    Conv1D(32, kernel_size=7, activation="relu"),
    MaxPooling1D(pool_size=2),

    Conv1D(64, kernel_size=5, activation="relu"),
    MaxPooling1D(pool_size=2),

    Flatten(),

    Dense(64, activation="relu"),
    Dropout(0.5),

    Dense(5, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

Model compiled successfully.


In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.7358 - loss: 0.8465 - val_accuracy: 0.7311 - val_loss: 0.5711
Epoch 2/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8231 - loss: 0.4150 - val_accuracy: 0.9104 - val_loss: 0.3268
Epoch 3/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8903 - loss: 0.3074 - val_accuracy: 0.9009 - val_loss: 0.3072
Epoch 4/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9039 - loss: 0.2587 - val_accuracy: 0.9033 - val_loss: 0.2736
Epoch 5/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9292 - loss: 0.1963 - val_accuracy: 0.9127 - val_loss: 0.2698
Epoch 6/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9393 - loss: 0.1819 - val_accuracy: 0.9127 - val_loss: 0.3148
Epoch 7/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9499 - loss: 0.1479 - val_accuracy: 0.9127 - val_loss: 0.2800
Epoch 8/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9575 - loss: 0.1206 - val_accuracy: 0.9127 - v

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - accuracy: 0.9358 - loss: 0.2593
Test loss: 0.2592603862285614
Test accuracy: 0.9358490705490112


In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
y_pred_classes = y_pred.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_classes,
    target_names=["Wake", "N1", "N2", "N3", "REM"]
))

17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step
              precision    recall  f1-score   support

        Wake       0.98      0.99      0.99       399
          N1       0.00      0.00      0.00        12
          N2       0.95      0.74      0.83        50
          N3       0.81      0.95      0.88        44
         REM       0.61      0.88      0.72        25

    accuracy                           0.94       530
   macro avg       0.67      0.71      0.68       530
weighted avg       0.93      0.94      0.93       530



In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights))

print("Class weights:")
for class_id, weight in class_weights.items():
    print(class_id, "→", weight)

Class weights:
0 → 0.26533166458072593
1 → 9.217391304347826
2 → 2.12
3 → 2.409090909090909
4 → 4.24


In [ ]:
history_weighted = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.9564 - loss: 0.2885 - val_accuracy: 0.9104 - val_loss: 0.3378
Epoch 2/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9428 - loss: 0.2879 - val_accuracy: 0.9033 - val_loss: 0.4041
Epoch 3/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9475 - loss: 0.2703 - val_accuracy: 0.9009 - val_loss: 0.3416
Epoch 4/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9452 - loss: 0.2334 - val_accuracy: 0.8939 - val_loss: 0.3472
Epoch 5/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9434 - loss: 0.2100 - val_accuracy: 0.8844 - val_loss: 0.3505
Epoch 6/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9605 - loss: 0.2019 - val_accuracy: 0.8868 - val_loss: 0.3829
Epoch 7/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9558 - loss: 0.1600 - val_accuracy: 0.9080 - val_loss: 0.4314
Epoch 8/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9670 - loss: 0.1348 - val_accuracy: 0.9009 - v

In [ ]:
test_loss_weighted, test_accuracy_weighted = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Weighted test loss:", test_loss_weighted)
print("Weighted test accuracy:", test_accuracy_weighted)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9170 - loss: 0.3408
Weighted test loss: 0.3407711982727051
Weighted test accuracy: 0.9169811606407166


In [ ]:
y_pred_weighted = model.predict(X_test)
y_pred_weighted_classes = y_pred_weighted.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_weighted_classes,
    target_names=["Wake", "N1", "N2", "N3", "REM"]
))

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
              precision    recall  f1-score   support

        Wake       0.99      0.98      0.98       399
          N1       0.00      0.00      0.00        12
          N2       0.80      0.74      0.77        50
          N3       0.78      0.91      0.84        44
         REM       0.51      0.76      0.61        25

    accuracy                           0.92       530
   macro avg       0.62      0.68      0.64       530
weighted avg       0.91      0.92      0.91       530



In [ ]:
from mne.datasets.sleep_physionet.age import fetch_data

files_more = fetch_data(
    subjects=list(range(5)),
    recording=[1]
)

print("Number of subjects downloaded:", len(files_more))

for i, subject_files in enumerate(files_more):
    print(i, subject_files)

Using default location ~/mne_data for PHYSIONET_SLEEP...


  0%|                                              | 0.00/51.1M [00:00<?, ?B/s]

  0%|                                              | 0.00/3.90k [00:00<?, ?B/s]

  0%|                                              | 0.00/51.1M [00:00<?, ?B/s]

  0%|                                              | 0.00/4.80k [00:00<?, ?B/s]

  0%|                                              | 0.00/51.4M [00:00<?, ?B/s]

  0%|                                              | 0.00/3.70k [00:00<?, ?B/s]

  0%|                                              | 0.00/46.9M [00:00<?, ?B/s]

  0%|                                              | 0.00/4.83k [00:00<?, ?B/s]

Download complete in 34m13s (191.3 MB)
Number of subjects downloaded: 5
0 ['/root/mne_data/physionet-sleep-data/SC4001E0-PSG.edf', '/root/mne_data/physionet-sleep-data/SC4001EC-Hypnogram.edf']
1 ['/root/mne_data/physionet-sleep-data/SC4011E0-PSG.edf', '/root/mne_data/physionet-sleep-data/SC4011EH-Hypnogram.edf']
2 ['/root/mne_data/physionet-sleep-data/SC4021E0-PSG.edf', '/root/mne_data/physionet-sleep-data/SC4021EH-Hypnogram.edf']
3 ['/root/mne_data/physionet-sleep-data/SC4031E0-PSG.edf', '/root/mne_data/physionet-sleep-data/SC4031EC-Hypnogram.edf']
4 ['/root/mne_data/physionet-sleep-data/SC4041E0-PSG.edf', '/root/mne_data/physionet-sleep-data/SC4041EC-Hypnogram.edf']


In [ ]:
def create_dataset_from_subject(subject_files):
    psg_file = subject_files[0]
    hypnogram_file = subject_files[1]

    # Load PSG
    raw = mne.io.read_raw_edf(
        psg_file,
        preload=True,
        verbose=False
    )

    # Read hypnogram
    annotations = mne.read_annotations(hypnogram_file)

    # Extract EEG
    eeg = raw.get_data(
        picks=["EEG Fpz-Cz"]
    )[0]

    sampling_rate = int(raw.info["sfreq"])
    samples_per_epoch = sampling_rate * 30
    recording_duration = len(eeg) / sampling_rate

    stage_mapping = {
        "Sleep stage W": 0,
        "Sleep stage 1": 1,
        "Sleep stage 2": 2,
        "Sleep stage 3": 3,
        "Sleep stage 4": 3,
        "Sleep stage R": 4
    }

    X_subject = []
    y_subject = []

    for onset, duration, description in zip(
        annotations.onset,
        annotations.duration,
        annotations.description
    ):

        if description not in stage_mapping:
            continue

        label = stage_mapping[description]

        num_epochs = int(duration // 30)

        for j in range(num_epochs):

            start_time = onset + j * 30
            end_time = start_time + 30

            if end_time > recording_duration:
                continue

            start_sample = int(start_time * sampling_rate)
            end_sample = start_sample + samples_per_epoch

            epoch = eeg[start_sample:end_sample]

            if len(epoch) == samples_per_epoch:
                X_subject.append(epoch)
                y_subject.append(label)

    return np.array(X_subject), np.array(y_subject)

In [ ]:
all_X = []
all_y = []

for i, subject_files in enumerate(files_more):

    print(f"Processing subject {i}...")

    X_subject, y_subject = create_dataset_from_subject(subject_files)

    print(
        f"  X: {X_subject.shape}, "
        f"y: {y_subject.shape}"
    )

    all_X.append(X_subject)
    all_y.append(y_subject)

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)

print("\nFinal combined dataset:")
print("X_all:", X_all.shape)
print("y_all:", y_all.shape)

Processing subject 0...


/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(


  X: (2650, 3000), y: (2650,)
Processing subject 1...


/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(


  X: (2802, 3000), y: (2802,)
Processing subject 2...


...

  X: (2804, 3000), y: (2804,)
Processing subject 3...


/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(


  X: (2820, 3000), y: (2820,)
Processing subject 4...


/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3260552372.py:6: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(


  X: (2569, 3000), y: (2569,)

Final combined dataset:
X_all: (13645, 3000)
y_all: (13645,)


In [ ]:
subject_ids = []

for i, X_subject in enumerate(all_X):
    subject_ids.extend(
        [i] * len(X_subject)
    )

subject_ids = np.array(subject_ids)

print("Subject IDs shape:", subject_ids.shape)
print("First 20 IDs:", subject_ids[:20])
print("Last 20 IDs:", subject_ids[-20:])

Subject IDs shape: (13645,)
First 20 IDs: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Last 20 IDs: [4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4]


In [ ]:
train_mask = subject_ids < 4
test_mask = subject_ids == 4

X_train = X_all[train_mask]
y_train = y_all[train_mask]

X_test = X_all[test_mask]
y_test = y_all[test_mask]

print("Training subjects: 0, 1, 2, 3")
print("Test subject: 4")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

Training subjects: 0, 1, 2, 3
Test subject: 4
X_train: (11076, 3000)
y_train: (11076,)
X_test : (2569, 3000)
y_test : (2569,)


In [ ]:
# Calculate normalization statistics only from training data
mean = X_train.mean()
std = X_train.std()

# Normalize
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# Reshape for Conv1D
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("Training mean:", X_train.mean())
print("Training std :", X_train.std())


X_train: (11076, 3000, 1)
X_test : (2569, 3000, 1)
Training mean: -1.6984126877255914e-17
Training std : 1.0000000000000013


In [ ]:
from collections import Counter

print("Training distribution:")
print(Counter(y_train))

print("\nTest distribution:")
print(Counter(y_test))

Training distribution:
Counter({np.int64(0): 7768, np.int64(2): 1842, np.int64(4): 667, np.int64(3): 477, np.int64(1): 322})

Test distribution:
Counter({np.int64(0): 1534, np.int64(2): 620, np.int64(4): 196, np.int64(1): 166, np.int64(3): 53})


In [ ]:
classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights))

print("Class weights:")

for class_id, weight in class_weights.items():
    print(
        class_id,
        "→",
        round(weight, 3)
    )

Class weights:
0 → 0.285
1 → 6.88
2 → 1.203
3 → 4.644
4 → 3.321


In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)

model_multi = Sequential([
    Input(shape=(3000, 1)),

    Conv1D(32, kernel_size=7, activation="relu"),
    MaxPooling1D(pool_size=2),

    Conv1D(64, kernel_size=5, activation="relu"),
    MaxPooling1D(pool_size=2),

    Flatten(),

    Dense(64, activation="relu"),
    Dropout(0.5),

    Dense(5, activation="softmax")
])

model_multi.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_multi = model_multi.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.6757 - loss: 1.0560 - val_accuracy: 0.7870 - val_loss: 0.4968
Epoch 2/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.8192 - loss: 0.7098 - val_accuracy: 0.8741 - val_loss: 0.4091
Epoch 3/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8652 - loss: 0.5632 - val_accuracy: 0.8366 - val_loss: 0.5258
Epoch 4/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.8964 - loss: 0.4825 - val_accuracy: 0.8660 - val_loss: 0.2879
Epoch 5/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.9144 - loss: 0.4122 - val_accuracy: 0.8619 - val_loss: 0.3042
Epoch 6/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9222 - loss: 0.3442 - val_accuracy: 0.8466 - val_loss: 0.7150
Epoch 7/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.9344 - loss: 0.2842 - val_accuracy: 0.8624 - val_loss: 0.4470
Epoch 8/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.9379 - loss: 0.2558 - val_acc

In [ ]:
test_loss, test_accuracy = model_multi.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.8089 - loss: 1.5805
Test loss: 1.5805155038833618
Test accuracy: 0.8088750243186951


In [ ]:
y_pred_multi = model_multi.predict(X_test)
y_pred_multi_classes = y_pred_multi.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_multi_classes,
    target_names=["Wake", "N1", "N2", "N3", "REM"]
))

81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
              precision    recall  f1-score   support

        Wake       0.84      0.99      0.91      1534
          N1       0.26      0.31      0.28       166
          N2       0.91      0.79      0.85       620
          N3       0.88      0.13      0.23        53
         REM       0.67      0.01      0.02       196

    accuracy                           0.81      2569
   macro avg       0.71      0.45      0.46      2569
weighted avg       0.81      0.81      0.77      2569



In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

print("Early stopping configured.")

Early stopping configured.


In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    Flatten,
    Dense,
    Dropout
)

model_improved = Sequential([
    Input(shape=(3000, 1)),

    Conv1D(32, kernel_size=7, activation="relu"),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(64, kernel_size=5, activation="relu"),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(128, kernel_size=3, activation="relu"),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(5, activation="softmax")
])

model_improved.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_improved.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 2994, 32)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 2994, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 1497, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 1493, 64)       │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1493, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 746, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 744, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 744, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 372, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 47616)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │     6,094,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,131,781 (23.39 MB)

 Trainable params: 6,131,333 (23.39 MB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
history_improved = model_improved.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - accuracy: 0.7080 - loss: 2.8785 - val_accuracy: 0.5023 - val_loss: 1.8717
Epoch 2/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.7844 - loss: 1.0376 - val_accuracy: 0.8362 - val_loss: 0.4676
Epoch 3/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.8341 - loss: 0.8127 - val_accuracy: 0.8227 - val_loss: 0.4016
Epoch 4/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.7775 - loss: 0.9306 - val_accuracy: 0.8425 - val_loss: 0.3596
Epoch 5/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.8064 - loss: 0.8050 - val_accuracy: 0.7721 - val_loss: 0.4640
Epoch 6/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.8067 - loss: 0.7086 - val_accuracy: 0.8100 - val_loss: 0.4854
Epoch 7/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.8375 - loss: 0.7144 - val_accuracy: 0.8700 - val_loss: 0.3997


In [ ]:
test_loss_improved, test_accuracy_improved = model_improved.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Improved test loss:", test_loss_improved)
print("Improved test accuracy:", test_accuracy_improved)

81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.7672 - loss: 1.3250
Improved test loss: 1.3250097036361694
Improved test accuracy: 0.7672246098518372


In [ ]:
y_pred_improved = model_improved.predict(X_test)
y_pred_improved_classes = y_pred_improved.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_improved_classes,
    target_names=["Wake", "N1", "N2", "N3", "REM"]
))

81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
              precision    recall  f1-score   support

        Wake       0.86      0.99      0.92      1534
          N1       0.18      0.31      0.23       166
          N2       0.94      0.60      0.74       620
          N3       0.67      0.55      0.60        53
         REM       0.05      0.02      0.03       196

    accuracy                           0.77      2569
   macro avg       0.54      0.49      0.50      2569
weighted avg       0.77      0.77      0.75      2569



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred_multi_classes
)

print(cm)

[[1526    8    0    0    0]
 [ 113   52    1    0    0]
 [  39   88  491    1    1]
 [   1    0   45    7    0]
 [ 142   52    0    0    2]]


In [ ]:
raw_check = mne.io.read_raw_edf(
    files_more[0][0],
    preload=False,
    verbose=False
)

print(raw_check.ch_names)

['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']


/tmp/ipykernel_3994/1132597723.py:1: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw_check = mne.io.read_raw_edf(
/tmp/ipykernel_3994/1132597723.py:1: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw_check = mne.io.read_raw_edf(
/tmp/ipykernel_3994/1132597723.py:1: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw_check = mne.io.read_raw_edf(


In [ ]:
for channel in [
    "EEG Fpz-Cz",
    "EEG Pz-Oz",
    "EOG horizontal",
    "EMG submental"
]:
    print(
        channel,
        "→",
        raw_check.info["sfreq"],
        "Hz"
    )

EEG Fpz-Cz → 100.0 Hz
EEG Pz-Oz → 100.0 Hz
EOG horizontal → 100.0 Hz
EMG submental → 100.0 Hz


In [ ]:
def create_multichannel_dataset(subject_files):

    psg_file = subject_files[0]
    hypnogram_file = subject_files[1]

    # Load PSG
    raw = mne.io.read_raw_edf(
        psg_file,
        preload=True,
        verbose=False
    )

    # Read hypnogram
    annotations = mne.read_annotations(hypnogram_file)

    # Channels we want
    selected_channels = [
        "EEG Fpz-Cz",
        "EEG Pz-Oz",
        "EOG horizontal",
        "EMG submental"
    ]

    # Extract all 4 channels
    data = raw.get_data(
        picks=selected_channels
    )

    sampling_rate = int(raw.info["sfreq"])
    samples_per_epoch = sampling_rate * 30
    recording_duration = data.shape[1] / sampling_rate

    stage_mapping = {
        "Sleep stage W": 0,
        "Sleep stage 1": 1,
        "Sleep stage 2": 2,
        "Sleep stage 3": 3,
        "Sleep stage 4": 3,
        "Sleep stage R": 4
    }

    X_subject = []
    y_subject = []

    for onset, duration, description in zip(
        annotations.onset,
        annotations.duration,
        annotations.description
    ):

        if description not in stage_mapping:
            continue

        label = stage_mapping[description]

        num_epochs = int(duration // 30)

        for j in range(num_epochs):

            start_time = onset + j * 30
            end_time = start_time + 30

            if end_time > recording_duration:
                continue

            start_sample = int(start_time * sampling_rate)
            end_sample = start_sample + samples_per_epoch

            epoch = data[
                :,
                start_sample:end_sample
            ]

            if epoch.shape == (4, samples_per_epoch):

                # Change:
                # (4, 3000) → (3000, 4)
                epoch = epoch.T

                X_subject.append(epoch)
                y_subject.append(label)

    return np.array(X_subject), np.array(y_subject)

In [ ]:
all_X_multi = []
all_y_multi = []
all_subject_ids_multi = []

for i, subject_files in enumerate(files_more):

    print(f"Processing subject {i}...")

    X_subject, y_subject = create_multichannel_dataset(
        subject_files
    )

    print(
        f"  X: {X_subject.shape}, "
        f"y: {y_subject.shape}"
    )

    all_X_multi.append(X_subject)
    all_y_multi.append(y_subject)

    all_subject_ids_multi.extend(
        [i] * len(X_subject)
    )

X_multi = np.concatenate(
    all_X_multi,
    axis=0
)

y_multi = np.concatenate(
    all_y_multi,
    axis=0
)

subject_ids_multi = np.array(
    all_subject_ids_multi
)

print("\nFinal dataset:")
print("X_multi:", X_multi.shape)
print("y_multi:", y_multi.shape)
print("subject_ids:", subject_ids_multi.shape)

Processing subject 0...


/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(


  X: (2650, 3000, 4), y: (2650,)
Processing subject 1...


/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(


  X: (2802, 3000, 4), y: (2802,)
Processing subject 2...


...

  X: (2804, 3000, 4), y: (2804,)
Processing subject 3...


/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(


  X: (2820, 3000, 4), y: (2820,)
Processing subject 4...


/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(
/tmp/ipykernel_3994/3745276841.py:7: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(


  X: (2569, 3000, 4), y: (2569,)

Final dataset:
X_multi: (13645, 3000, 4)
y_multi: (13645,)
subject_ids: (13645,)


In [ ]:
train_mask_multi = subject_ids_multi < 4
test_mask_multi = subject_ids_multi == 4

X_train_multi = X_multi[train_mask_multi]
y_train_multi = y_multi[train_mask_multi]

X_test_multi = X_multi[test_mask_multi]
y_test_multi = y_multi[test_mask_multi]

print("X_train_multi:", X_train_multi.shape)
print("y_train_multi:", y_train_multi.shape)

print("X_test_multi :", X_test_multi.shape)
print("y_test_multi :", y_test_multi.shape)


X_train_multi: (11076, 3000, 4)
y_train_multi: (11076,)
X_test_multi : (2569, 3000, 4)
y_test_multi : (2569,)


In [ ]:
mean_multi = X_train_multi.mean(
    axis=(0, 1),
    keepdims=True
)

std_multi = X_train_multi.std(
    axis=(0, 1),
    keepdims=True
)

X_train_multi = (
    X_train_multi - mean_multi
) / std_multi

X_test_multi = (
    X_test_multi - mean_multi
) / std_multi

print("Mean shape:", mean_multi.shape)
print("Std shape :", std_multi.shape)

print("Training mean:", X_train_multi.mean(axis=(0, 1)))
print("Training std :", X_train_multi.std(axis=(0, 1)))

Mean shape: (1, 1, 4)
Std shape : (1, 1, 4)
Training mean: [ 5.74194671e-13 -2.43137184e-15  3.00339795e-14 -6.99701465e-14]
Training std : [1. 1. 1. 1.]


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes_multi = np.unique(y_train_multi)

weights_multi = compute_class_weight(
    class_weight="balanced",
    classes=classes_multi,
    y=y_train_multi
)

class_weights_multi = dict(
    zip(classes_multi, weights_multi)
)

print("Class weights:")

for class_id, weight in class_weights_multi.items():
    print(class_id, "→", round(weight, 3))

Class weights:
0 → 0.285
1 → 6.88
2 → 1.203
3 → 4.644
4 → 3.321


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

model_multi_channel = Sequential([

    Input(shape=(3000, 4)),

    Conv1D(
        32,
        kernel_size=7,
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(
        64,
        kernel_size=5,
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(
        128,
        kernel_size=3,
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    GlobalAveragePooling1D(),

    Dense(64, activation="relu"),
    Dropout(0.4),

    Dense(5, activation="softmax")
])

model_multi_channel.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_9 (Conv1D)               │ (None, 2994, 32)       │           928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 2994, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 1497, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 1493, 64)       │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1493, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 746, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 744, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 744, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 372, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,413 (177.39 KB)

 Trainable params: 44,965 (175.64 KB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
model_multi_channel.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

Model compiled successfully.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping_multi = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_multi = model_multi_channel.fit(
    X_train_multi,
    y_train_multi,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights_multi,
    callbacks=[early_stopping_multi],
    verbose=1
)

Epoch 1/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 19s 40ms/step - accuracy: 0.8518 - loss: 0.5617 - val_accuracy: 0.8290 - val_loss: 0.4923
Epoch 2/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.8947 - loss: 0.4279 - val_accuracy: 0.8222 - val_loss: 0.5187
Epoch 3/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9160 - loss: 0.3778 - val_accuracy: 0.8917 - val_loss: 0.3341
Epoch 4/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9194 - loss: 0.3594 - val_accuracy: 0.8633 - val_loss: 0.3697
Epoch 5/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9178 - loss: 0.3289 - val_accuracy: 0.8538 - val_loss: 0.3793
Epoch 6/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9272 - loss: 0.3206 - val_accuracy: 0.9152 - val_loss: 0.2612
Epoch 7/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9264 - loss: 0.3198 - val_accuracy: 0.8488 - val_loss: 0.4493
Epoch 8/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9317 - loss: 0.3044 - val_acc

In [ ]:
test_loss_multi, test_accuracy_multi = model_multi_channel.evaluate(
    X_test_multi,
    y_test_multi,
    verbose=1
)

print("Test loss:", test_loss_multi)
print("Test accuracy:", test_accuracy_multi)

81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8295 - loss: 0.5572
Test loss: 0.5572348833084106
Test accuracy: 0.8295056223869324


In [ ]:
from sklearn.metrics import classification_report

y_pred_multi_channel = model_multi_channel.predict(
    X_test_multi,
    verbose=1
)

y_pred_multi_channel_classes = np.argmax(
    y_pred_multi_channel,
    axis=1
)

print(
    classification_report(
        y_test_multi,
        y_pred_multi_channel_classes,
        target_names=[
            "Wake",
            "N1",
            "N2",
            "N3",
            "REM"
        ],
        digits=2
    )
)

81/81 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step
              precision    recall  f1-score   support

        Wake       1.00      0.92      0.96      1534
          N1       0.38      0.81      0.52       166
          N2       0.96      0.56      0.71       620
          N3       0.62      0.89      0.73        53
         REM       0.52      0.94      0.67       196

    accuracy                           0.83      2569
   macro avg       0.69      0.83      0.72      2569
weighted avg       0.91      0.83      0.84      2569



In [ ]:
from sklearn.metrics import confusion_matrix

cm_multi_channel = confusion_matrix(
    y_test_multi,
    y_pred_multi_channel_classes
)

print(cm_multi_channel)

[[1415   93    2   14   10]
 [   0  135    4    1   26]
 [   0  122  349   14  135]
 [   0    0    6   47    0]
 [   0    8    3    0  185]]
